In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 3.12 The Relativistic Formulation of Maxwell's Equations

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume III — Classical Electrodynamics",
    number="3.12",
    title="The Relativistic Formulation of Maxwell's Equations",
    blurb="The summit: spacetime, the field tensor, and Maxwell's four equations in "
    "two lines. Electricity and magnetism turn out to be a single object seen from "
    "different frames — magnetism is what an electric field looks like to a moving "
    "observer.",
    difficulty="advanced",
    estimate="180–220 min",
)

## Notebook overview

This is the longest and most demanding notebook of the volume, and the most rewarding.
It asks two unfamiliar things at once, to think in four dimensions and to think in
tensors, and in return the eleven notebooks before it snap into a single structure. The
claim is audacious and exact: **electricity and magnetism are not two phenomena but
one**, a single geometric object, and which part of it you call "electric" and which
"magnetic" depends only on how you are moving. Magnetism is what an electric field looks
like to a moving observer.

A note on reading order: this notebook sits in Volume III as the capstone of
electrodynamics, but it leans on special relativity. If you are meeting relativity for
the first time, you may prefer to read the special-relativity notebooks of Volume IV
([§4.1](../04-special-relativity/crisis-and-postulates.ipynb)–[§4.5](../04-special-relativity/four-momentum-energy.ipynb))
first and return here afterwards. We develop just enough relativity inline for
the notebook to stand on its own, but the fuller story is in Volume IV. In particular
the four-vector and `np.einsum` machinery we use inline here (the metric $\eta$, the
contraction $\eta_{\mu\nu}a^\mu b^\nu$, raising and lowering) is developed carefully, from
the geometry up, in [§4.3](../04-special-relativity/spacetime-minkowski.ipynb); a
reader who wants the index-string grammar spelled out
should look there.

The road there starts from the crisis [§3.8](maxwell-waves.ipynb) left open. Maxwell's
equations pick out a
speed $c=1/\sqrt{\mu_0\varepsilon_0}$, but a speed *relative to what*? Einstein's answer,
that $c$ is the same in every inertial frame and space and time themselves transform to
make it so, forces the Lorentz transformation, four-vectors, and a four-dimensional
spacetime. We develop only as much special relativity as we need to rewrite
electrodynamics; the full development is Volume IV, which we cross-reference. With that
groundwork we build the **field tensor** $F^{\mu\nu}$, the six numbers of $\mathbf E$
and $\mathbf B$ assembled into one antisymmetric object, collapse all four Maxwell
equations into **two tensor lines**, and watch a boost mix $\mathbf E$ and $\mathbf B$
into each other. We end with the two Lorentz invariants every observer agrees on, and
with the gauge arc's final turn: the freedom of [§3.6](magnetostatics.ipynb) and the
tool of [§3.8](maxwell-waves.ipynb) are revealed as
**structure**, $F^{\mu\nu}=\partial^\mu A^\nu-\partial^\nu A^\mu$ being manifestly
gauge-invariant, with a forward pointer to where gauge invariance becomes physics (Vol
V) and back to its Noether root ([§2.2](../02-classical-mechanics/noether.ipynb)).

We work in **SI units** with the metric signature $(-,+,+,+)$, used
throughout. (At this altitude Gaussian units are tidier, hiding the factors of $c$; we
keep SI for continuity and note where $c$ would vanish.) The one animation is genuinely
warranted: a boost continuously sweeping a pure electric field into a mixture of
electric and magnetic, the unification set in motion. Everything else is a still.

> **How to read the checks.** Each exercise ends with a `validate` call against an
> independent fact: the spacetime interval invariant under a boost, the four-velocity
> norm $-c^2$, the field tensor antisymmetric, the covariant equations reproducing the
> four 3-vector ones, the field invariants frame-independent, $F^{\mu\nu}$ unchanged by a
> gauge transformation. A ✓ is strong evidence; a ✗ is a prompt to *locate the
> discrepancy*, not a verdict.
>
> **Scope.** Enough relativity to recast electrodynamics, not a relativity course (that
> is Vol IV). See Nolting, *Theoretical Physics 3/4* {cite}`nolting3`; Griffiths,
> *Introduction to Electrodynamics* {cite}`griffiths_em` (ch. 12); Jackson
> {cite}`jackson` (ch. 11); Landau & Lifshitz, *The Classical Theory of Fields*
> {cite}`ll2`.

## Theory in brief

### The crisis that forces relativity

Maxwell's equations give one speed $c=1/\sqrt{\mu_0\varepsilon_0}$
([§3.8](maxwell-waves.ipynb)), but Galilean
velocity addition makes every speed frame-dependent. In which frame, then, is light's
speed $c$? The nineteenth-century answer, a luminiferous aether, failed (Michelson–
Morley). Einstein's resolution is the postulate

```{math}
:label: eq-crisis
c \text{ is the same in every inertial frame,}
```

from which space and time must themselves transform.

### Spacetime and the Lorentz transformation

Events live in four-dimensional spacetime, $x^\mu=(ct,x,y,z)$, with the invariant
interval set by the Minkowski metric $\eta=\mathrm{diag}(-1,1,1,1)$,

```{math}
:label: eq-lorentz
s^2 = \eta_{\mu\nu}x^\mu x^\nu = -c^2t^2+x^2+y^2+z^2 .
```

A boost along $x$ at speed $v$ ($\beta=v/c$, $\gamma=1/\sqrt{1-\beta^2}$) is the linear
map $\Lambda$ that mixes $t$ and $x$ while leaving $s^2$ unchanged.

### Four-vectors

Quantities that transform like $x^\mu$ are four-vectors: the four-velocity
$u^\mu=\gamma(c,\mathbf v)$ with the invariant norm $u^\mu u_\mu=-c^2$, and the two that
carry electrodynamics,

```{math}
:label: eq-four-vectors
J^\mu=(c\rho,\mathbf J), \qquad A^\mu=(V/c,\mathbf A).
```

Charge conservation and the Lorenz gauge each become a single four-divergence,
$\partial_\mu J^\mu=0$ and $\partial_\mu A^\mu=0$.

### The field tensor

The six numbers in $\mathbf E$ and $\mathbf B$ are not two 3-vectors but the six
independent components of one antisymmetric rank-2 tensor,

```{math}
:label: eq-field-tensor
F^{\mu\nu}=\partial^\mu A^\nu-\partial^\nu A^\mu, \qquad
F^{0i}=E_i/c, \quad F^{ij}=-\varepsilon^{ijk}B_k .
```

Written as a $4\times4$ matrix it holds $\mathbf E$ in its time row/column and $\mathbf
B$ in its spatial block: $\mathbf E$ and $\mathbf B$ are one object. Griffiths,
*Introduction to Electrodynamics*, ch. 12, works the identification of the components
out entry by entry.

### Maxwell in two lines

All four equations collapse to the sourced pair and the source-free (Bianchi) pair,

```{math}
:label: eq-covariant-maxwell
\partial_\mu F^{\mu\nu}=\mu_0 J^\nu, \qquad
\partial^\alpha F^{\beta\gamma}+\partial^\beta F^{\gamma\alpha}+\partial^\gamma F^{\alpha\beta}=0 .
```

The first reproduces Gauss ($\nu=0$) and Ampère–Maxwell ($\nu=i$); the second reproduces
$\nabla\cdot\mathbf B=0$ and Faraday.

### E and B mix under boosts

Transforming the tensor to a boosted frame, $F'^{\mu\nu}=\Lambda^\mu{}_\alpha
\Lambda^\nu{}_\beta F^{\alpha\beta}$, mixes the fields,

```{math}
:label: eq-EB-mixing
E'_\parallel=E_\parallel,\quad \mathbf E'_\perp=\gamma(\mathbf E+\mathbf v\times\mathbf B)_\perp,\quad
\mathbf B'_\perp=\gamma\Big(\mathbf B-\tfrac{1}{c^2}\mathbf v\times\mathbf E\Big)_\perp .
```

A pure Coulomb field, seen from a moving frame, acquires a magnetic component:
**magnetism is electricity in motion**.

### Gauge invariance as structure, and the invariants

Because $\partial^\mu\partial^\nu\chi$ is symmetric, $F^{\mu\nu}=\partial^\mu A^\nu-
\partial^\nu A^\mu$ is **manifestly unchanged** under

```{math}
:label: eq-gauge-structure
A^\mu \to A^\mu+\partial^\mu\chi ,
```

so the gauge freedom of [§3.6](magnetostatics.ipynb) and the gauge tool of
[§3.8](maxwell-waves.ipynb) are now seen as a structural
feature of the four-potential, with the Lorenz condition $\partial_\mu A^\mu=0$
manifestly Lorentz-invariant. From $F^{\mu\nu}$ one builds two Lorentz **scalars**,

```{math}
:label: eq-invariants
\mathbf E\cdot\mathbf B \;\propto\; F_{\mu\nu}\widetilde F^{\mu\nu}, \qquad
E^2-c^2B^2 \;\propto\; F_{\mu\nu}F^{\mu\nu},
```

the same in every frame even as $\mathbf E$ and $\mathbf B$ separately change. Landau &
Lifshitz, *The Classical Theory of Fields*, construct the dual tensor
$\widetilde F^{\mu\nu}$ and derive both invariants.

## Setup

Data and instruments only: the CODATA constants that fix $c=1/\sqrt{\mu_0\varepsilon_0}$,
the series palette, the Minkowski metric $\eta=\mathrm{diag}(-1,1,1,1)$ that fixes this
notebook's signature, and the metric contraction $\eta_{\mu\nu}a^\mu b^\nu$ written once
as a call so the exercises read as physics rather than as index strings. This notebook's
own machinery is *not* here: you write the Lorentz boost $\Lambda(\beta)$ in Exercise 1,
the four-velocity $u^\mu$ in Exercise 2, and the field tensor $F^{\mu\nu}$ together with
its inverse read-out in Exercise 3. Every boost, every mixing of $\mathbf E$ and $\mathbf
B$, and every invariant downstream runs on those three. No randomness appears anywhere in
this notebook.

The Setup below holds this notebook's data and instruments — nothing you
are asked to build. It is collapsed so the building stays yours; expand it
whenever you want the details.

<!-- setup-policy: v2 -->

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

from ecp import draw, validate
from ecp.animate import show

# data: CODATA vacuum permeability and permittivity (via scipy.constants), and the
# speed of light they fix — the constant whose frame-independence forces relativity
from scipy.constants import mu_0 as MU0  # vacuum permeability, T·m/A
from scipy.constants import epsilon_0 as EPS0  # vacuum permittivity, F/m

C_LIGHT = 1.0 / np.sqrt(MU0 * EPS0)  # speed of light, m/s

# data: the series palette
ACCENT, INK, SOFT = draw.ACCENT, draw.INK, draw.SOFT

# data: the Minkowski metric, the signature convention (−,+,+,+) this notebook fixes
ETA = np.diag([-1.0, 1.0, 1.0, 1.0])


# instrument: the metric contraction spelled out once as a call, so that the exercises
# read as physics rather than as index strings — it is a single `np.einsum` and nobody's
# lesson here; the index-string grammar itself is developed from the geometry up in
# [§4.3](../04-special-relativity/spacetime-minkowski.ipynb).
def minkowski_dot(a, b):
    """Minkowski inner product a^μ η_μν b^ν in signature (−,+,+,+).

    Parameters
    ----------
    a, b : array_like
        Four-vectors.

    Returns
    -------
    float
        The invariant scalar product (``np.einsum`` over the metric).
    """
    return float(np.einsum("m,mn,n->", a, ETA, b))

## Movement I — Spacetime and four-vectors

## Exercise 1 — The Lorentz transformation and the invariant interval (worked)

Special relativity begins by taking $c$ as absolute {eq}`eq-crisis` and letting space
and time bend to keep it so. The carrier of that bending is the **Lorentz boost**
$\Lambda$, which mixes $t$ and $x$ but preserves the interval $s^2=\eta_{\mu\nu}x^\mu
x^\nu$ {eq}`eq-lorentz` ({numref}`fig-rm-spacetime`). What looks like a fixed time and a
fixed position to one observer is a blend of both to another, yet both agree on $s^2$.
The boost here runs at $v=0.6c$, and the three events it acts on include one sitting on
the light line $ct=x$: whatever the boost does, that event must stay on the light line,
because $c$ is the same in both frames.

1. Write `lorentz_boost(beta)`, returning the $4\times4$ matrix $\Lambda$ of a boost
   along $x$ at $\beta=v/c$: $\gamma=1/\sqrt{1-\beta^2}$ in the $(0,0)$ and $(1,1)$
   slots, $-\gamma\beta$ in the $(0,1)$ and $(1,0)$ slots, and the identity on $y$ and
   $z$, which a boost along $x$ leaves alone. **Write this one yourself** — the
   implementation is the lesson, and every change of frame in this notebook goes
   through it.
2. Apply it to a set of events by matrix multiplication, `events @ Λ.T`, and form the
   interval as the contraction `x @ η @ x` (the Setup's `minkowski_dot`) before and
   after, confirming $s^2$ is unchanged while $t$ and $x$ individually transform.
3. Check that the light-line event maps to another light-line event: the speed of light
   is $c$ in both frames, the whole point.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.close(
    s2_after, s2_before, "the spacetime interval is Lorentz-invariant", rtol=1e-10
)
validate.close(
    light_after[0],
    light_after[1],
    "a light ray maps to a light ray — c is the same in both frames",
    rtol=1e-10,
)

## Exercise 2 — Four-vectors: velocity, current, potential (worked)

Anything that transforms under $\Lambda$ like the position $x^\mu$ is a **four-vector**,
and the Minkowski norm $a^\mu a_\mu$ is then a scalar every observer agrees on. The
four-velocity $u^\mu=\gamma(c,\mathbf v)$ {eq}`eq-four-vectors` has the fixed norm
$u^\mu u_\mu=-c^2$, a compact way of saying everything advances through spacetime at
speed $c$. The same packaging gives the **four-current** $J^\mu=(c\rho,\mathbf J)$ and
**four-potential** $A^\mu=(V/c,\mathbf A)$, and in this language two scattered facts of
[§3.8](maxwell-waves.ipynb), charge conservation and the Lorenz gauge, each become a
single four-divergence.

The four-potential used below is a transverse plane wave, $A^\mu=a^\mu\cos(kx-\omega t)$
with $a^\mu=(0,0,a_y,0)$ and $\omega=ck$: because $a^\mu$ has no time or $x$ component,
$\partial_\mu A^\mu=(1/c)\partial_t A^0+\partial_x A^1$ vanishes identically, so the
Lorenz condition is satisfied and the numerics should see it.

1. Write `four_velocity(v)`, returning $u^\mu=\gamma(c,\mathbf v)$
   {eq}`eq-four-vectors` for an ordinary 3-velocity $\mathbf v$, with
   $\gamma=1/\sqrt{1-\mathbf v\cdot\mathbf v/c^2}$.
2. For $v=0.6c$ along $x$, verify $u^\mu u_\mu=-c^2$ by contracting it with the metric,
   `np.einsum("m,mn,n->", u, η, u)` (the Setup's `minkowski_dot`).
3. Form the four-divergence $\partial_\mu A^\mu$ of that plane-wave four-potential by
   differencing each component with `numpy.gradient` along its own axis and summing,
   confirming the Lorenz condition holds — the single equation $\partial_\mu A^\mu=0$
   standing in for $\nabla\cdot\mathbf A+\tfrac1{c^2}\partial_t V=0$.

In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.close(
    u_norm, -(C_LIGHT**2), "the four-velocity has invariant norm −c²", rtol=1e-10
)
validate.close(
    lorenz_residual,
    0.0,
    "the Lorenz gauge ∂_μA^μ = 0 is a single four-divergence",
    atol=1e-6,
)

## Movement II — The field tensor and Maxwell

## Exercise 3 — The field tensor $F^{\mu\nu}$ (worked)

Here is the centerpiece. The six numbers we have carried as two separate 3-vectors,
$\mathbf E$ and $\mathbf B$, are really the six independent entries of one antisymmetric
$4\times4$ tensor $F^{\mu\nu}$ {eq}`eq-field-tensor`, with $\mathbf E$ along the time
row and column and $\mathbf B$ in the spatial block ({numref}`fig-rm-tensor`). The
antisymmetry $F^{\mu\nu}=-F^{\nu\mu}$ leaves exactly six free components, precisely the
count of $\mathbf E$ and $\mathbf B$ together. They were never two things. The field the
packing is exercised on is $\mathbf E=(3,-1,2)\times10^5\,$V/m and $\mathbf
B=(0,4,-2)\times10^{-4}\,$T, a generic field with no special alignment.

1. Write `field_tensor(E, B)`, assembling those six numbers into one $4\times4$ array
   {eq}`eq-field-tensor`: $E_i/c$ across the time row, $-E_i/c$ down the time column,
   the magnetic components filling the spatial block antisymmetrically ($F^{12}=B_z$,
   $F^{23}=B_x$, $F^{31}=B_y$, and their negatives across the diagonal), and zeros on
   the diagonal itself. **Write this one yourself** — the implementation is the lesson,
   and this object is what the rest of the notebook is about.
2. Write `extract_EB(F)`, the inverse read-out: $E_i=c\,F^{0i}$, and each $B_k$ from the
   spatial slot it was packed into. **Write this one yourself** — the implementation is
   the lesson, and the read-out only inverts if the packing was right.
3. Confirm the antisymmetry with `np.allclose(F, -F.T)` (read off here as
   $\max|F+F^{\mathsf T}|$) and send $\mathbf E$ and $\mathbf B$ through both directions
   for a clean round trip. Six components, one object.

In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.close(
    antisym_residual,
    0.0,
    "the field tensor is antisymmetric — E and B are its six components",
    atol=1e-12,
)
validate.close(
    np.concatenate([E_back, B_back]),
    np.concatenate([E_test, B_test]),
    "E and B round-trip through F^μν exactly",
    rtol=1e-12,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 4 — Maxwell's equations in covariant form (worked)

Now the collapse. The two tensor equations {eq}`eq-covariant-maxwell` contain all four
of Maxwell's laws. The sourced equation $\partial_\mu F^{\mu\nu}=\mu_0 J^\nu$ gives
Gauss's law for $\nu=0$ and Ampère–Maxwell for $\nu=i$; the Bianchi identity (the
source-free pair) gives $\nabla\cdot\mathbf B=0$ and Faraday. Eleven notebooks of
separate laws, written in two lines.

What we check here is that unpacking, not a solution: the test field is an arbitrary
smooth $\mathbf E(t,\mathbf r)$, $\mathbf B(t,\mathbf r)$ on a $16^4$ grid over the unit
spacetime box, and it need not satisfy Maxwell at all, because the identity between the
tensor form and the 3-vector form holds field by field. Written out, the two components
we test read $\partial_\mu F^{\mu0}=-(\nabla\cdot\mathbf E)/c$ (Gauss) and $\partial_\mu
F^{\mu i}=(\nabla\times\mathbf B-\tfrac1{c^2}\partial_t\mathbf E)_i$ (Ampère–Maxwell).

1. Assemble $F^{\mu\nu}$ on the 4-D grid: the same packing you wrote in Exercise 3, but
   with each entry now a whole array, so the $4\times4$ is written out as a nested list
   rather than built by a call.
2. Write `tensor_divergence(nu)`, forming $\partial_\mu F^{\mu\nu}$ by differentiating
   each $F^{\mu\nu}$ along its own $x^\mu$ axis with `numpy.gradient` — and since
   $x^0=ct$, the time derivative is $\partial_0=\partial/\partial(ct)$ — then summing
   over $\mu$. **Write this one yourself** — the implementation is the lesson: the whole
   content of a covariant equation is which index runs against which axis.
3. Confirm component by component that it equals the corresponding 3-vector operator,
   to finite-difference precision on the interior (the boundary nodes use one-sided
   differences, so they are excluded). The tensor equation *is* the 3-vector equations.

In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.close(
    divF_0[core],
    gauss_3v[core],
    "∂_μF^μ0 = μ₀J⁰ reproduces Gauss's law (∇·E component-wise)",
    atol=1e-9,
)
validate.close(
    divF_1[core],
    ampere_x_3v[core],
    "∂_μF^μi = μ₀J^i reproduces Ampère–Maxwell (component-wise)",
    atol=1e-9,
)

## Movement III — The unification: E and B mix

## Exercise 5 — Boosting the field tensor (worked)

Because $F^{\mu\nu}$ is a tensor, a change of frame acts on it by the boost on each
index, $F'^{\mu\nu}=\Lambda^\mu{}_\alpha\Lambda^\nu{}_\beta F^{\alpha\beta}$, i.e.
$F'=\Lambda F\Lambda^{\mathsf T}$ {eq}`eq-EB-mixing`. Reading $\mathbf E'$ and $\mathbf
B'$ out of the boosted tensor must reproduce the textbook transformation rules, with the
parallel components unchanged and the perpendicular ones mixing $\mathbf E$ and $\mathbf
B$. For a boost along $x$ at $v=0.6c$ those rules are $E'_x=E_x$,
$E'_y=\gamma(E_y-vB_z)$, $E'_z=\gamma(E_z+vB_y)$, and $B'_x=B_x$,
$B'_y=\gamma(B_y+vE_z/c^2)$, $B'_z=\gamma(B_z-vE_y/c^2)$ — derived by hand in the
textbooks, one component at a time.

1. Boost the $F^{\mu\nu}$ of Exercise 3 by contracting both indices with the
   `lorentz_boost` you wrote in Exercise 1: `np.einsum('ma,nb,ab->mn', Λ, Λ, F)`,
   equivalently $\Lambda F\Lambda^{\mathsf T}$.
2. Read $\mathbf E'$ and $\mathbf B'$ out of the result with your Exercise 3
   `extract_EB` and check them against the closed-form rules above. Tensor algebra and
   the hand-derived formulas must agree to machine precision.

In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.close(
    Ep_tensor,
    Ep_formula,
    "the boosted field tensor reproduces the E transformation law",
    rtol=1e-8,
)
validate.close(
    Bp_tensor,
    Bp_formula,
    "the boosted field tensor reproduces the B transformation law",
    rtol=1e-8,
)

## Exercise 6 — Magnetism is electricity in another frame (worked)

This is the volume's deepest result. Take a **pure electric field**, $\mathbf B=0$, the
Coulomb field of a static charge, and look at it from a moving frame. A genuine
**magnetic** field appears: $\mathbf B'\neq0$ {eq}`eq-EB-mixing`. There is no separate
"magnetism" waiting in the wings; the magnetic field of a moving charge
([§3.6](magnetostatics.ipynb)) simply
*is* its electric field, seen by an observer in motion ({numref}`fig-rm-boost-anim`). The
animation that closes the exercise sweeps the boost from $0$ to $0.9c$ and watches
$\mathbf B'$ grow from nothing as the frame speeds up, the unification set in motion.

1. Build the pure-$E$ field tensor from $\mathbf E=(0,10^5,0)\,$V/m and $\mathbf B=0$
   with the `field_tensor` you wrote in Exercise 3, apply the perpendicular boost at
   $v=0.6c$ with the same `np.einsum('ma,nb,ab->mn', Λ, Λ, F)` contraction as
   Exercise 5, and read $\mathbf B'$ back out with your Exercise 3 `extract_EB`.
2. Show the magnetic field that emerges equals the closed form
   $B'_z=-\gamma vE_y/c^2$.
3. Sweep $\beta$ from $0$ to $0.9$, rebuilding $\Lambda(\beta)$ each time with your
   Exercise 1 `lorentz_boost`, and animate $E'_y$ and $B'_z$ against $\beta$
   ({numref}`fig-rm-boost-anim`).

In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
validate.close(
    Bp_pure[2],
    -g * v * E_pure[1] / C_LIGHT**2,
    "a pure electric field acquires exactly B'_z = −γvE_y/c² under a boost — "
    "magnetism is relativity",
    rtol=1e-8,
)
validate.close(
    np.array([Bp_pure[0], Bp_pure[1]]),
    np.zeros(2),
    "the boost generates no magnetic component along x or y (geometry of v × E)",
    atol=1e-15,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 7 — The current-carrying wire, demystified (student)

The most famous demonstration that magnetism is relativity is a neutral wire. In the lab
it carries a current and exerts a **magnetic** force on a nearby moving charge. But in
the charge's own rest frame there is no motion of the charge to feel a magnetic force;
instead, the wire's positive and negative charge densities Lorentz-contract by different
amounts, so the wire appears **charged**, and the same force arrives as an **electric**
one. Two descriptions, one physics.

The model is positive ions at rest and electrons drifting at $v_d$, each species with
proper line density $\lambda_0=10^{-8}\,$C/m, so the wire is neutral in the lab; the test
charge is $q=1\,$nC moving parallel to the wire at $u=5\times10^5\,$m/s, a distance
$r=0.02\,$m away, and the drift is $v_d=10^5\,$m/s. Two facts from earlier do the work:
the wire's field $B=\mu_0 I/2\pi r$ ([§3.6](magnetostatics.ipynb)) and the line charge's
field $E=\lambda/2\pi\varepsilon_0 r$ ([§3.3](gauss-law.ipynb)). One relativistic
subtlety: a transverse force is not itself invariant — it picks up a factor $\gamma_u$
between the lab and the charge's rest frame, so the two forces are compared as
$F'/\gamma_u$ against $F$.

1. In the lab, compute the current $I=\lambda_0 v_d$ and the magnetic force $F=quB$ on
   the test charge.
2. Transform to the charge's rest frame: velocity-add the electron drift and the charge's
   own motion, $w_e=(v_d+u)/(1+v_du/c^2)$, and Lorentz-contract each species' line
   density by its own $\gamma$ — the ions now move at $u$, the electrons at $w_e$ — to
   get the net density $\lambda_{\rm net}$, which is no longer zero.
3. Compute the rest-frame electric force $F'=qE'$ with $E'=\lambda_{\rm
   net}/2\pi\varepsilon_0 r$, and confirm $F'/\gamma_u$ reproduces the lab's magnetic
   force.

In [ ]:
# (solution hidden on the public site)


### Validation 7

In [ ]:
validate.close(
    F_elec_rest / gu,
    F_mag_lab,
    "the lab magnetic force equals the electric force in the charge's rest frame",
    rtol=1e-3,
)

## Movement IV — Invariants and structure

## Exercise 8 — The Lorentz invariants (worked)

Although $\mathbf E$ and $\mathbf B$ each change from frame to frame, two combinations
of them do not {eq}`eq-invariants`: the scalars $\mathbf E\cdot\mathbf B$ and
$E^2-c^2B^2$, built from $F_{\mu\nu}\widetilde F^{\mu\nu}$ and $F_{\mu\nu}F^{\mu\nu}$,
are the same in every frame. Whether a field is "more electric" or "more magnetic" is a
matter of who is looking, but these two numbers are absolute, and they classify the
field: a radiation field, for instance, has both invariants zero for every observer.

1. Write `invariants(E, B)`, returning the pair $(\mathbf E\cdot\mathbf B,\;E^2-c^2B^2)$
   as the dot product `E @ B` and the combination `E @ E - c**2 * (B @ B)`.
2. Evaluate it on the lab-frame $\mathbf E,\mathbf B$ of Exercise 3 and on the boosted
   $\mathbf E',\mathbf B'$ of Exercise 5, and confirm both numbers are unchanged even
   though $\mathbf E$ and $\mathbf B$ themselves are not.

In [ ]:
# (solution hidden on the public site)


### Validation 8

In [ ]:
validate.close(
    np.array([EdotB_boost, inv2_boost]),
    np.array([EdotB_lab, inv2_lab]),
    "the two field invariants E·B and E²−c²B² are the same in all frames",
    rtol=1e-6,
)

## Exercise 9 — Gauge invariance as structure (worked)

The gauge arc reaches its summit. Because $F^{\mu\nu}=\partial^\mu A^\nu-\partial^\nu
A^\mu$ and partial derivatives commute, adding the gradient of any scalar to the
four-potential, $A^\mu\to A^\mu+\partial^\mu\chi$ {eq}`eq-gauge-structure`, changes
nothing: $\partial^\mu\partial^\nu\chi-\partial^\nu\partial^\mu\chi=0$. The gauge freedom
that was a *freedom* in [§3.6](magnetostatics.ipynb) and a *tool* in
[§3.8](maxwell-waves.ipynb) is now seen as built into the very
definition of the field, **structure**. And the Lorenz condition $\partial_\mu A^\mu=0$,
being a four-divergence, is manifestly Lorentz-invariant, where the Coulomb gauge
$\nabla\cdot\mathbf A=0$ is not.

The demonstration runs on the $(ct,x)$ sector alone, where a single component
$F^{tx}=\partial^tA^x-\partial^xA^t$ carries the whole story; with the signature
$(-,+,+,+)$, raising the time index flips its sign, $\partial^t=-\partial_{ct}$, while
$\partial^x=\partial_x$. The four-potential is $A^0=\sin x\cos(ct)$, $A^1=\cos x\sin(ct)$
and the gauge scalar is $\chi=0.7\sin(x+ct)+0.3\cos 2x$, chosen for no reason at all —
which is the point, since *any* $\chi$ must leave $F$ alone.

1. Write `Fxt(A0, A1)`, building $F^{tx}$ from a four-potential on the grid by
   differencing with `numpy.gradient` and raising both indices with the metric.
   **Write this one yourself** — the implementation is the lesson: the sign that the
   raised time index puts on $\partial_{ct}$ is the whole content of "raising an index".
2. Evaluate $F^{tx}$ for that four-potential, then shift it by
   $A^\mu\to A^\mu+\partial^\mu\chi$ (again via `numpy.gradient`, with the same raising
   rule), rebuild $F^{tx}$, and confirm the two agree with `np.allclose` — here read off
   as the maximum difference on the interior, where the central differences are clean.

The arc closes: **freedom ([§3.6](magnetostatics.ipynb)) → tool
([§3.8](maxwell-waves.ipynb)) → structure (here) → physics (Vol VI, Aharonov–Bohm)**,
rooted in the symmetry–conservation link of Noether
([§2.2](../02-classical-mechanics/noether.ipynb)), where a global phase symmetry gives
charge conservation and its local version is gauge symmetry itself.

In [ ]:
# (solution hidden on the public site)


### Validation 9

In [ ]:
validate.close(
    F_after[core2],
    F_before[core2],
    "F^μν is invariant under a gauge transformation A → A + ∂χ (gauge as structure)",
    atol=1e-9,
)

## Exercise 10 — What the volume built

Stand at the summit and look back. The volume opened with a single static charge and the
question of what it does to the space around it ([§3.1](coulomb-field.ipynb)). It
found a field, then a potential, then a local law (Gauss, [§3.3](gauss-law.ipynb)); it
added magnetism ([§3.6](magnetostatics.ipynb)), coupled the two
through induction ([§3.7](induction.ipynb)), and closed the set with the displacement
current into Maxwell's
equations and the discovery that light is an electromagnetic wave
([§3.8](maxwell-waves.ipynb)). It confined
those waves ([§3.9](waveguides-cavities.ipynb)), found what produces them
([§3.10](radiation.ipynb)), and met them again as circuits
([§3.11](rlc-circuits.ipynb)). And here, at the end, all of it, eleven notebooks of
separate laws, collapses
into two tensor equations for a single object $F^{\mu\nu}$ in a four-dimensional
spacetime. Electrodynamics is revealed as one **relativistic field theory**, the first
and the exemplar of classical field theory.

The crisis that drove us, "$c$ relative to what?", is resolved: there is no preferred
frame, $\mathbf E$ and $\mathbf B$ are one object, and the laws read the same for every
observer. The road continues to special relativity in full (Vol IV), to quantum
electrodynamics where gauge invariance becomes physics (Vol VI), and to field theory
beyond. The volume began by asking what a charge does to the space around it, and ends
by finding that space, time, electricity, and magnetism are one structure.

1. Confirm the summit numerically in one line: the same field configuration, viewed in
   the lab and in the boosted frame of Exercise 5, has different $\mathbf E$ and $\mathbf
   B$ but identical invariants and an identical, antisymmetric $F^{\mu\nu}$ structure —
   the single object underneath the two appearances.

In [ ]:
# (solution hidden on the public site)


### Validation 10

In [ ]:
validate.check(
    one_field,
    "one electromagnetic field: frame-independent invariants and an antisymmetric F^μν",
)

## Notebook summary

- **Spacetime and four-vectors.** The boost $\Lambda(0.6c)$ preserves the interval
  $s^2=\eta_{\mu\nu}x^\mu x^\nu$ and maps light rays to light rays ($c$ invariant); the
  four-velocity has norm $u^\mu u_\mu=-c^2$, and the Lorenz gauge is the single
  four-divergence $\partial_\mu A^\mu=0$.
- **The field tensor.** $\mathbf E$ and $\mathbf B$ are the six components of one
  antisymmetric $F^{\mu\nu}$ {eq}`eq-field-tensor` (round-tripped exactly), and Maxwell's
  four equations are the two tensor lines $\partial_\mu F^{\mu\nu}=\mu_0 J^\nu$ and the
  Bianchi identity, verified component-by-component against $\nabla\cdot\mathbf E$ and
  $\nabla\times\mathbf B-\tfrac1{c^2}\partial_t\mathbf E$.
- **The unification.** Boosting $F^{\mu\nu}$ reproduces the $\mathbf E,\mathbf B$
  transformation laws to machine precision; a **pure electric field acquires a magnetic
  one** under a boost ($B'_z=-\gamma vE_y/c^2$, animated), and the current-carrying wire's
  magnetic force is an electric force in the charge's frame. Magnetism is electricity in
  motion.
- **Invariants and structure.** $\mathbf E\cdot\mathbf B$ and $E^2-c^2B^2$ are the same
  in every frame; $F^{\mu\nu}=\partial^\mu A^\nu-\partial^\nu A^\mu$ is manifestly
  gauge-invariant, closing the arc **freedom ([§3.6](magnetostatics.ipynb)) → tool
  ([§3.8](maxwell-waves.ipynb)) → structure (here) →
  physics (Vol VI)**, rooted in Noether
  ([§2.2](../02-classical-mechanics/noether.ipynb)). From a static charge
  ([§3.1](coulomb-field.ipynb)) to one
  relativistic field theory: **Volume III complete**.

## Outlook

- **Special relativity in full (Vol IV).** Kinematics, dynamics, $E=mc^2$, and the
  spacetime geometry developed here only as far as electrodynamics required.
- **The stress–energy tensor.** $F^{\mu\nu}$ also builds $T^{\mu\nu}$, the energy,
  momentum, and stress carried by the field itself, the source of gravity in general
  relativity.
- **Relativistic radiation.** The Liénard–Wiechert potentials extend the radiation of
  [§3.10](radiation.ipynb) to fast charges, giving synchrotron light and the
  relativistic beaming of
  accelerators.
- **Gauge theory and the Standard Model.** Promoting the global phase symmetry of
  Noether ([§2.2](../02-classical-mechanics/noether.ipynb)) to a local one *is*
  electromagnetism; generalising the group leads to
  Yang–Mills theory and the Standard Model, far beyond this course's arc.
- **Quantum electrodynamics (Vol VI).** Gauge invariance becomes physical in the
  Aharonov–Bohm effect, and minimal coupling weds the four-potential to the quantum
  particle, where this volume's structure becomes the language of modern physics.

In [ ]:
from ecp.style import footer

footer()